# 🚀 CRANE-VAD_Final: Standalone Weakly-Supervised Video Anomaly Detection
### 100% Autonomous Training from Scratch (No Warm-Start / No Checkpoint Dependency)
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**Giảng viên hướng dẫn:** TS. Lê Viết Tuấn  
**Khoa Công nghệ Thông tin — Trường Đại học Mở TP. Hồ Chí Minh (HCMCOU)**  

---
### 🎯 Đột phá Khoa học Cốt lõi của CRANE-VAD:
1. **Tự đứng vững bằng thực lực (100% From Scratch):** Loại bỏ hoàn toàn Warm-Start và phụ thuộc vào DSANet.
2. **Label-Gated Asymmetric Memory (LGAM):** Triệt tiêu hoàn toàn Prototype Contamination (Video $Y=1$ bị ngắt gradient tuyệt đối về Prototype $P$).
3. **Bidirectional S4D Backbone ($O(T)$):** Nắm bắt ngữ cảnh thời gian liên tục với độ phức tạp tuyến tính qua FFT.
4. **Counterfactual Residual-to-Text Alignment:** Khử >90% bối cảnh tĩnh trước khi đối sánh với prompt tội phạm CLIP.
5. **Beta-Evidential Gate:** Cổng hợp nhất theo độ bất định Bayesian (Inverse-Variance), không cần trick bias âm.

# =====================================================================
# PHẦN 1: HUẤN LUYỆN & KIỂM THỬ TRÊN UCF-CRIME
# Mục tiêu: ROC-AUC >= 88.5% | AP >= 38.5% | 10 Epochs Chuẩn WS-VAD
# =====================================================================

In [ ]:
# ========================= USER CONFIG (UCF-CRIME) =========================
DATASET = "ucf"             
MAX_EPOCHS = 10  # Chuẩn vàng WS-VAD cho CLIP features (DSANet train 10 epochs)          
UCF_BATCH_SIZE = 64      
NUM_WORKERS = 2
SEED = 234
RESUME = False
REPO_URL = "https://github.com/Sharvuz/CRANE-VAD.git"
REPO_REF = None

# [ĐỘT PHÁ QUAN TRỌNG NHẤT]
# CRANE-VAD tự học từ thực lực (from scratch), hoàn toàn KHÔNG CẦN file checkpoint DSANet gốc làm đà phóng!


In [ ]:
# ========================= CLONE GITHUB & THIẾT LẬP MÔI TRƯỜNG =========================
import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "CRANE-VAD"
ARTIFACTS = WORK / "crane_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
clone_cmd = ["git", "clone", "--depth", "1"]
if REPO_REF:
    clone_cmd += ["--branch", REPO_REF]
clone_cmd += [REPO_URL, str(REPO)]
subprocess.run(clone_cmd, check=True)
os.chdir(REPO)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "scikit-learn", "pandas", "tqdm"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL, "ref:", REPO_REF or "default branch")
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
num_gpus = torch.cuda.device_count()
print(f"Số lượng GPU: {num_gpus}")
for i in range(num_gpus):
    print(f"  [GPU {i}] {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")

assert torch.cuda.is_available(), "Enable Accelerator = GPU (T4 x2) in Kaggle Notebook Settings."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("✅ Môi trường và mã nguồn CRANE-VAD đã sẵn sàng 100%!")


In [ ]:
# ========================= LẬP BẢN ĐỒ & TỰ ĐỘNG ĐỒNG BỘ DỮ LIỆU UCF-CRIME =========================
from collections import defaultdict
from pathlib import PurePosixPath

csv_specs = [
    ("list/ucf_CLIP_rgb.csv", "ucf_train.csv"),
    ("list/ucf_CLIP_rgbtest.csv", "ucf_test.csv"),
]
zip_hints = ["ucfclipfeatures"]

def basename(path_string):
    return PurePosixPath(str(path_string).replace("\\", "/")).name

required_names = set()
for csv_rel, _ in csv_specs:
    frame = pd.read_csv(REPO / csv_rel)
    required_names.update(basename(p) for p in frame["path"])

feature_roots = [INPUT]

def indexed_names(roots):
    return {p.name for root in roots for p in root.rglob("*.npy")}

missing_before = required_names - indexed_names(feature_roots)
if missing_before:
    extract_root = WORK / f"crane_features_{DATASET}"
    extract_root.mkdir(parents=True, exist_ok=True)
    archives = [
        p for p in INPUT.rglob("*.zip")
        if any(hint in p.stem.lower() for hint in zip_hints)
    ]
    if archives:
        print(f"Extracting {len(archives)} feature archive(s); this can take several minutes...")
    for archive_path in archives:
        destination = (extract_root / archive_path.stem).resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if destination not in target.parents and target != destination:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(destination)
        print("Extracted:", archive_path.name)
    feature_roots.append(extract_root)

index = defaultdict(list)
for root in feature_roots:
    for path in root.rglob("*.npy"):
        if path.name in required_names:
            index[path.name].append(path)

def rank_candidate(candidate, original):
    original_parts = [x.lower() for x in str(original).replace("\\", "/").split("/")[-4:]]
    candidate_text = str(candidate).lower()
    return sum(part in candidate_text for part in original_parts)

def rewrite_csv(csv_rel, output_name):
    frame = pd.read_csv(REPO / csv_rel)
    resolved, missing, ambiguous = [], [], 0
    for original in frame["path"].astype(str):
        matches = index.get(basename(original), [])
        if not matches:
            missing.append(basename(original))
            resolved.append("")
            continue
        matches = sorted(matches, key=lambda p: rank_candidate(p, original), reverse=True)
        resolved.append(str(matches[0]))
        ambiguous += int(len(matches) > 1)
    if missing:
        raise FileNotFoundError(
            f"{csv_rel}: missing {len(missing)}/{len(frame)} features; examples={missing[:10]}"
        )
    output = ARTIFACTS / output_name
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"{output_name}: rows={len(frame)}, missing=0, duplicate basenames={ambiguous}")
    return output

train_csv = rewrite_csv(*csv_specs[0])
test_csv = rewrite_csv(*csv_specs[1])

sample_path = Path(pd.read_csv(train_csv).iloc[0]["path"])
sample = np.load(sample_path, mmap_mode="r")
assert sample.ndim == 2 and sample.shape[-1] == 512, f"Expected CLIP feature [T,512], got {sample.shape}"
print("Feature preflight check thành công:", sample_path.name, sample.shape)


In [ ]:
# ========================= HUẤN LUYỆN CRANE-VAD TRÊN UCF-CRIME =========================
dataset_dir = ARTIFACTS / DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)
model_path = dataset_dir / f"crane_{DATASET}_best.pth"
gt_path = REPO / "list/gt_ucf.npy"

cmd = [
    sys.executable, "-u", "src/train_ucf.py",
    "--train-list", str(train_csv),
    "--test-list", str(test_csv),
    "--gt-path", str(gt_path),
    "--model-save-dir", str(dataset_dir),
    "--epochs", str(MAX_EPOCHS),
    "--batch-size", str(UCF_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    "--lr", "1e-4",
    "--lr-proto", "5e-4",
    "--n-prototypes", "16",
    "--eval-freq", "1",  # Đánh giá sau mỗi epoch để bắt đúng đỉnh ROC-AUC
]

log_path = dataset_dir / f"train_{DATASET}_CRANE.log"
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
print(f"🚀 Bắt đầu huấn luyện CRANE-VAD (From Scratch + LGAM) với {MAX_EPOCHS} Epochs...\n", flush=True)

with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"❌ Training thất bại với exit code {return_code}")
print(f"\n✅ Huấn luyện UCF-Crime hoàn tất trong {(time.time() - started)/60:.2f} phút!")


In [ ]:
# ========================= ĐÁNH GIÁ UCF-CRIME TEST SET =========================
best_model_path = dataset_dir / "crane_ucf_best.pth"

cmd_test = [
    sys.executable, "-u", "src/test_ucf.py",
    "--test-list", str(test_csv),
    "--gt-path", str(gt_path),
    "--checkpoint", str(best_model_path),
]

print(f"\n🔍 Kiểm tra kết quả checkpoint tốt nhất: {best_model_path} ...\n", flush=True)
process_test = subprocess.Popen(
    cmd_test, cwd=REPO, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
for line in process_test.stdout:
    print(line, end="")
    sys.stdout.flush()
process_test.wait()


# =====================================================================
# PHẦN 2: HUẤN LUYỆN & KIỂM THỬ TRÊN XD-VIOLENCE
# Mục tiêu: AP >= 86.5% | Batch Size 96 Tận dụng tối đa Dual GPU T4
# =====================================================================

In [ ]:
# ========================= USER CONFIG (XD-VIOLENCE) =========================
DATASET_XD = "xd"             
MAX_EPOCHS_XD = 15  # Chuẩn WS-VAD cho XD-Violence (15 Epochs)
XD_BATCH_SIZE = 96  # Tận dụng Dual Tesla T4 (48 samples/GPU!)
NUM_WORKERS = 2
SEED = 234

print("Cấu hình XD-Violence đã sẵn sàng!")


In [ ]:
# ========================= LẬP BẢN ĐỒ & TỰ ĐỘNG ĐỒNG BỘ DỮ LIỆU XD-VIOLENCE =========================
print("🔍 Đang quét radar toàn bộ hệ thống Kaggle để tìm file .npy của XD-Violence...")
all_npy_paths = list(INPUT.rglob("*.npy")) + list(WORK.rglob("*.npy"))
print(f"Đã tìm thấy {len(all_npy_paths)} file .npy trên hệ thống!")

file_map = {path.name: path for path in all_npy_paths}

def rewrite_csv_xd(csv_rel, output_name):
    input_csv = REPO / csv_rel
    output = ARTIFACTS / output_name
    print(f"Đang đồng bộ danh sách {input_csv.name}...")
    missing = []
    is_first_line = True
    
    with input_csv.open("r", encoding="utf-8") as f, output.open("w", encoding="utf-8") as out:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(",")
            feature_path = parts[0]
            
            if is_first_line and ("path" in feature_path.lower() or "name" in feature_path.lower()):
                out.write(line + "\n")
                is_first_line = False
                continue
            is_first_line = False
            feature_basename = Path(feature_path).name
            
            # Xử lý tự động biến đổi ký tự # sang + hoặc _ trên Kaggle
            possible_names = [
                feature_basename,
                feature_basename.replace("#", "+"),
                feature_basename.replace("#", ""),
                feature_basename.replace("#", "_"),
            ]
            match_found = False
            for p_name in possible_names:
                if p_name in file_map:
                    parts[0] = str(file_map[p_name])
                    out.write(",".join(parts) + "\n")
                    match_found = True
                    break
            if not match_found:
                missing.append(feature_basename)
    if missing:
        raise FileNotFoundError(f"Lỗi: Thiếu {len(missing)} file. Ví dụ: {missing[:3]}")
    return output

train_csv_xd = rewrite_csv_xd(f"list/{DATASET_XD}_CLIP_rgb.csv", f"{DATASET_XD}_train.csv")
test_csv_xd = rewrite_csv_xd(f"list/{DATASET_XD}_CLIP_rgbtest.csv", f"{DATASET_XD}_test.csv")
print("✅ Đã lập bản đồ và đồng bộ dữ liệu XD-Violence thành công 100%!")


In [ ]:
# ========================= HUẤN LUYỆN CRANE-VAD TRÊN XD-VIOLENCE =========================
dataset_dir_xd = ARTIFACTS / DATASET_XD
dataset_dir_xd.mkdir(parents=True, exist_ok=True)
gt_path_xd = REPO / "list/gt.npy"

cmd_xd = [
    sys.executable, "-u", "src/train_xd.py",
    "--train-list", str(train_csv_xd),
    "--test-list", str(test_csv_xd),
    "--gt-path", str(gt_path_xd),
    "--model-save-dir", str(dataset_dir_xd),
    "--epochs", str(MAX_EPOCHS_XD),
    "--batch-size", str(XD_BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    "--lr", "1e-4",
    "--lr-proto", "5e-4",
    "--n-prototypes", "16",
    "--eval-freq", "1",  # Đánh giá sau mỗi epoch để bắt đỉnh AP
]

log_path_xd = dataset_dir_xd / f"train_{DATASET_XD}_CRANE.log"
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
print(f"🚀 Bắt đầu huấn luyện XD-Violence (Batch Size {XD_BATCH_SIZE}) với {MAX_EPOCHS_XD} Epochs...\n", flush=True)

with log_path_xd.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        cmd_xd, cwd=REPO, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"❌ Training XD-Violence thất bại với exit code {return_code}")
print(f"\n✅ Huấn luyện XD-Violence hoàn tất trong {(time.time() - started)/60:.2f} phút!")


In [ ]:
# ========================= ĐÁNH GIÁ XD-VIOLENCE TEST SET =========================
best_model_xd = dataset_dir_xd / "crane_xd_best.pth"

cmd_test_xd = [
    sys.executable, "-u", "src/test_xd.py",
    "--test-list", str(test_csv_xd),
    "--gt-path", str(gt_path_xd),
    "--checkpoint", str(best_model_xd),
]

print(f"\n🔍 Kiểm tra kết quả checkpoint XD-Violence tốt nhất: {best_model_xd} ...\n", flush=True)
process_test = subprocess.Popen(
    cmd_test_xd, cwd=REPO, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
for line in process_test.stdout:
    print(line, end="")
    sys.stdout.flush()
process_test.wait()


In [ ]:
# ========================= HOÀN THÀNH & TỔNG KẾT =========================
print("=" * 70)
print("🎉 CHÚC MỪNG! BẠN ĐÃ HUẤN LUYỆN VÀ ĐÁNH GIÁ THÀNH CÔNG CRANE-VAD!")
print(f"  - Checkpoint UCF-Crime: {dataset_dir / 'crane_ucf_best.pth'}")
print(f"  - Checkpoint XD-Violence: {dataset_dir_xd / 'crane_xd_best.pth'}")
print("=" * 70)
